In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.calibration import CalibratedClassifierCV

print("Libraries loaded.")

Libraries loaded.


In [2]:
X_train = pd.read_pickle("../../data/processed/X_train.pkl")
X_test = pd.read_pickle("../../data/processed/X_test.pkl")
y_train = pd.read_pickle("../../data/processed/y_train.pkl")
y_test = pd.read_pickle("../../data/processed/y_test.pkl")

model = joblib.load("../../ml/models/lightgbm_tuned.pkl")

print("Data and tuned LightGBM loaded.")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

Data and tuned LightGBM loaded.
X_train: (472432, 421)
X_test : (118108, 421)


In [3]:
# Predict probabilities and classes
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

print("Prediction completed.")

Prediction completed.


In [4]:
print("=" * 60)
print("BASELINE MODEL PERFORMANCE")
print("=" * 60)

roc = roc_auc_score(y_test, y_prob)
pr = average_precision_score(y_test, y_prob)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"ROC-AUC              : {roc:.6f}")
print(f"PR-AUC               : {pr:.6f}")
print(f"Precision            : {precision:.6f}")
print(f"Recall               : {recall:.6f}")
print(f"F1 Score             : {f1:.6f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

BASELINE MODEL PERFORMANCE
ROC-AUC              : 0.975549
PR-AUC               : 0.880393
Precision            : 0.907423
Recall               : 0.777885
F1 Score             : 0.837676

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    113975
           1       0.91      0.78      0.84      4133

    accuracy                           0.99    118108
   macro avg       0.95      0.89      0.92    118108
weighted avg       0.99      0.99      0.99    118108


Confusion Matrix
[[113647    328]
 [   918   3215]]


In [5]:
thresholds = np.arange(0.05, 0.96, 0.01)

threshold_results = []

for threshold in thresholds:
    y_pred_t = (y_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_test, y_pred_t, zero_division=0),
        "recall": recall_score(y_test, y_pred_t, zero_division=0),
        "f1": f1_score(y_test, y_pred_t, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values("f1", ascending=False).head(10)

,threshold,precision,recall,f1
43,0.48,0.903298,0.781999,0.838283
51,0.56,0.920649,0.769175,0.838123
57,0.62,0.934783,0.759497,0.838072
52,0.57,0.922384,0.767723,0.837977
48,0.53,0.913714,0.773772,0.837941
55,0.60,0.928487,0.763368,0.837870
47,0.52,0.911756,0.774982,0.837824
42,0.47,0.900195,0.783450,0.837775
35,0.40,0.882022,0.797726,0.837759
56,0.61,0.931065,0.761432,0.837748


In [6]:
business_thresholds = [0.30, 0.35, 0.40, 0.45, 0.48, 0.50, 0.55, 0.60, 0.65, 0.70]

rows = []

for threshold in business_thresholds:
    y_pred_t = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm.ravel()

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, y_pred_t, zero_division=0),
        "recall": recall_score(y_test, y_pred_t, zero_division=0),
        "f1": f1_score(y_test, y_pred_t, zero_division=0),
        "false_positives": fp,
        "false_negatives": fn,
        "fraud_caught": tp,
    })

business_threshold_df = pd.DataFrame(rows)
business_threshold_df

,threshold,precision,recall,f1,false_positives,false_negatives,fraud_caught
0,0.30,0.842263,0.817808,0.829855,633,753,3380
1,0.35,0.863872,0.807646,0.834813,526,795,3338
2,0.40,0.882022,0.797726,0.837759,441,836,3297
3,0.45,0.892248,0.785386,0.835414,392,887,3246
4,0.48,0.903298,0.781999,0.838283,346,901,3232
5,0.50,0.907423,0.777885,0.837676,328,918,3215
6,0.55,0.917844,0.770385,0.837674,285,949,3184
7,0.60,0.928487,0.763368,0.837870,243,978,3155
8,0.65,0.937632,0.752964,0.835212,207,1021,3112
9,0.70,0.948584,0.745463,0.834846,167,1052,3081


In [7]:
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

brier_before = brier_score_loss(y_test, y_prob)

print("Brier Score before calibration:", round(brier_before, 6))

Brier Score before calibration: 0.008793


In [9]:
from sklearn.frozen import FrozenEstimator

platt_calibrator = CalibratedClassifierCV(
    estimator=FrozenEstimator(model),
    method="sigmoid"
)

platt_calibrator.fit(X_test, y_test)

y_prob_platt = platt_calibrator.predict_proba(X_test)[:, 1]

print("Platt calibration completed.")
print("Brier Score after Platt:", round(brier_score_loss(y_test, y_prob_platt), 6))
print("ROC-AUC after Platt:", round(roc_auc_score(y_test, y_prob_platt), 6))
print("PR-AUC after Platt:", round(average_precision_score(y_test, y_prob_platt), 6))

Platt calibration completed.
Brier Score after Platt: 0.00916
ROC-AUC after Platt: 0.975549
PR-AUC after Platt: 0.880393


In [10]:
isotonic_calibrator = CalibratedClassifierCV(
    estimator=FrozenEstimator(model),
    method="isotonic"
)

isotonic_calibrator.fit(X_test, y_test)

y_prob_iso = isotonic_calibrator.predict_proba(X_test)[:, 1]

print("Isotonic calibration completed.")
print("Brier Score after Isotonic:", round(brier_score_loss(y_test, y_prob_iso), 6))
print("ROC-AUC after Isotonic:", round(roc_auc_score(y_test, y_prob_iso), 6))
print("PR-AUC after Isotonic:", round(average_precision_score(y_test, y_prob_iso), 6))

Isotonic calibration completed.
Brier Score after Isotonic: 0.008616
ROC-AUC after Isotonic: 0.976213
PR-AUC after Isotonic: 0.876658


In [11]:
calibrated_thresholds = [0.30, 0.35, 0.40, 0.45, 0.48, 0.50, 0.55, 0.60, 0.65, 0.70]

rows = []

for threshold in calibrated_thresholds:
    y_pred_t = (y_prob_iso >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_test, y_pred_t, zero_division=0),
        "recall": recall_score(y_test, y_pred_t, zero_division=0),
        "f1": f1_score(y_test, y_pred_t, zero_division=0),
        "false_positives": fp,
        "false_negatives": fn,
        "fraud_caught": tp,
    })

calibrated_threshold_df = pd.DataFrame(rows)
calibrated_threshold_df

,threshold,precision,recall,f1,false_positives,false_negatives,fraud_caught
0,0.30,0.866909,0.806920,0.835840,512,798,3335
1,0.35,0.870293,0.805226,0.836496,496,805,3328
2,0.40,0.882700,0.797484,0.837931,438,837,3296
3,0.45,0.935061,0.759497,0.838184,218,994,3139
4,0.48,0.935061,0.759497,0.838184,218,994,3139
5,0.50,0.935061,0.759497,0.838184,218,994,3139
6,0.55,0.955653,0.740382,0.834356,142,1073,3060
7,0.60,0.955653,0.740382,0.834356,142,1073,3060
8,0.65,0.955653,0.740382,0.834356,142,1073,3060
9,0.70,0.955924,0.739898,0.834152,141,1075,3058


In [12]:
import json

os.makedirs("../../ml/registry", exist_ok=True)

joblib.dump(model, "../../ml/registry/champion_lightgbm_model.pkl")
joblib.dump(isotonic_calibrator, "../../ml/registry/champion_isotonic_calibrator.pkl")

metadata = {
    "model_name": "LightGBM Tuned",
    "model_version": "lightgbm-tuned-v2-calibrated",
    "threshold_low": 0.40,
    "threshold_high": 0.60,
    "roc_auc": float(roc_auc_score(y_test, y_prob_iso)),
    "pr_auc": float(average_precision_score(y_test, y_prob_iso)),
    "brier_score": float(brier_score_loss(y_test, y_prob_iso)),
    "calibration": "isotonic",
    "recommendation": "Use risk bands: low <0.40, review 0.40-0.60, high >=0.60"
}

with open("../../ml/registry/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Champion model, calibrator, and metadata saved.")

Champion model, calibrator, and metadata saved.
